# Refloxide refit kernel comparison

Load a pickled DFT ``GlobalObjective``, migrate it to refloxide
``BatchedGlobalObjective``, and compare three evaluation paths in one process:

1. **Stock pyref** on the original pickle (pure Python uniaxial kernel)
2. **Refloxide Python** (``use_rust=False``) on the converted batched objective
3. **Refloxide Rust** (``use_rust=True``, ``parallel=False``) on the same batched objective

Requires refloxide on the kernel (``make develop`` in the sibling repo).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from refloxide.integrations.pyref import pyref_patched
from utils import read_fit, read_ooc
from utils.refloxide_fitting import compare_kernel_paths, convert_dft_fit_bundle

if pyref_patched():
    raise RuntimeError(
        "Restart the kernel before running this notebook: stock pyref logl "
        "must be evaluated before refloxide patches pyref.fitting."
    )

In [ ]:
bundle = read_fit("dft/dft_en_offset_new2.pkl", material="znpc", source="local")
ooc_df = read_ooc("dft.csv", material="znpc")
len(bundle.objectives), bundle.logl()

In [ ]:
model, batched = convert_dft_fit_bundle(bundle, ooc_df)
len(batched.terms), len(batched.varying_parameters())

In [ ]:
result = compare_kernel_paths(bundle, batched, parallel=False)

summary = pd.DataFrame(
    {
        "path": ["pyref_stock", "refloxide_python", "refloxide_rust"],
        "logl": [
            result.pyref_stock_logl,
            result.refloxide_python_logl,
            result.refloxide_rust_logl,
        ],
    }
)
summary

In [ ]:
print(f"Python/Rust logl delta: {result.python_rust_logl_delta:.6e}")
print(
    f"Max |R_py - R_rust| at {result.diagnostic_energy_ev} eV pol={result.diagnostic_pol}: "
    f"{result.max_abs_reflectivity_delta_python_rust:.6e}"
)
pd.DataFrame(
    {
        "pyref_stock": result.pyref_stock_reflectivity[:8],
        "refloxide_python": result.refloxide_python_reflectivity[:8],
        "refloxide_rust": result.refloxide_rust_reflectivity[:8],
    }
)